### External

In [4]:
import kagglehub
import os
import zipfile
import os
from pathlib import Path
import shutil
from datasets import load_dataset

In [5]:


# Descargar dataset
path = kagglehub.dataset_download("kaustubhdhote/human-faces-dataset")

# Ruta destino
destino = Path("../data/external")

destino.mkdir(parents=True, exist_ok=True)

# Copiar archivos al proyecto
shutil.copytree(path, destino, dirs_exist_ok=True)

print("Archivos copiados correctamente")

100%|██████████| 114M/114M [00:08<00:00, 14.7MB/s] 

Extracting files...


Archivos copiados correctamente


In [6]:
# ==========================================
# DESCARGAR CELEBA DESDE HUGGINGFACE
# ==========================================


PROJECT_ROOT = Path().resolve().parent

CELEBA_PATH = PROJECT_ROOT / "data" / "external" / "celeba"
CELEBA_PATH.mkdir(parents=True, exist_ok=True)

celeba_dataset = load_dataset("nielsr/CelebA-faces", split="train")

print(f"CelebA size: {len(celeba_dataset)}")

c:\Users\maria\Desktop\DL\autoencoder-project\venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\maria\.cache\huggingface\hub\datasets--nielsr--CelebA-faces. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating train split: 100%|██████████| 202599/202599 [00:03<00:00, 66068.52 examples/s]


CelebA size: 202599


### Raw

In [7]:
from pathlib import Path
from PIL import Image
import shutil

In [8]:
# ========= 1. Definir rutas =========

PROJECT_ROOT = Path().resolve().parent
EXTERNAL_PATH = PROJECT_ROOT / "data" / "external" / "human faces dataset"
RAW_PATH = PROJECT_ROOT / "data" / "raw"

AI_SRC = EXTERNAL_PATH / "ai-generated images"
REAL_SRC = EXTERNAL_PATH / "real images"

AI_DST = RAW_PATH / "ai"
REAL_DST = RAW_PATH / "real"

AI_DST.mkdir(parents=True, exist_ok=True)
REAL_DST.mkdir(parents=True, exist_ok=True)

# ========= 2. Función para validar imágenes =========

def process_images(src_folder, dst_folder):
    total = 0
    valid = 0
    corrupted = 0
    
    for img_path in src_folder.glob("*"):
        total += 1
        
        try:
            with Image.open(img_path) as img:
                img.verify()  # verifica que no esté corrupta
            
            shutil.copy(img_path, dst_folder / img_path.name)
            valid += 1
            
        except Exception:
            corrupted += 1
    
    return total, valid, corrupted

# ========= 3. Procesar datasets =========

ai_total, ai_valid, ai_corrupted = process_images(AI_SRC, AI_DST)
real_total, real_valid, real_corrupted = process_images(REAL_SRC, REAL_DST)

# ========= 4. Reporte simple =========

print("===== DATASET REPORT =====")
print(f"AI Images     → Total: {ai_total}, Válidas: {ai_valid}, Corruptas: {ai_corrupted}")
print(f"Real Images   → Total: {real_total}, Válidas: {real_valid}, Corruptas: {real_corrupted}")
print(f"Total limpio en RAW: {ai_valid + real_valid}")

===== DATASET REPORT =====
AI Images     → Total: 4630, Válidas: 4630, Corruptas: 0
Real Images   → Total: 5000, Válidas: 5000, Corruptas: 0
Total limpio en RAW: 9630


In [9]:
print("\nBalance de clases:")
print(f"% AI   : {ai_valid / (ai_valid + real_valid) * 100:.2f}%")
print(f"% Real : {real_valid / (ai_valid + real_valid) * 100:.2f}%")


Balance de clases:
% AI   : 48.08%
% Real : 51.92%


In [10]:
# ==========================================
# GUARDAR CELEBA COMO REAL EXTRA
# ==========================================

CELEBA_RAW = RAW_PATH / "celeba_real"
CELEBA_RAW.mkdir(parents=True, exist_ok=True)

max_images = 50000  # opcional para no usar las 200k

for i, example in enumerate(celeba_dataset):
    if i >= max_images:
        break
    
    img = example["image"].convert("RGB")
    img.save(CELEBA_RAW / f"celeba_{i}.jpg")

print(f"Saved {max_images} CelebA images to RAW")

Saved 50000 CelebA images to RAW


### Interim

In [11]:
import os
import random
import shutil
from pathlib import Path


In [12]:

# Para reproducibilidad
random.seed(42)

PROJECT_ROOT = Path().resolve().parent

RAW_PATH = PROJECT_ROOT / "data" / "raw"
INTERIM_PATH = PROJECT_ROOT / "data" / "interim"

train_split = 0.7
val_split = 0.15
test_split = 0.15

for split in ["train", "val", "test"]:
    for cls in ["ai", "real"]:
        (INTERIM_PATH / split / cls).mkdir(parents=True, exist_ok=True)

def split_class_images_multi(source_dirs, class_name):
    all_images = []
    
    for source_dir in source_dirs:
        all_images.extend(list(source_dir.glob("*")))
    
    random.shuffle(all_images)
    
    total = len(all_images)
    train_size = int(total * train_split)
    val_size = int(total * val_split)
    
    train_imgs = all_images[:train_size]
    val_imgs = all_images[train_size:train_size + val_size]
    test_imgs = all_images[train_size + val_size:]
    
    splits = {
        "train": train_imgs,
        "val": val_imgs,
        "test": test_imgs
    }
    
    for split_name, img_list in splits.items():
        for img_path in img_list:
            shutil.copy(
                img_path,
                INTERIM_PATH / split_name / class_name / img_path.name
            )
    
    return total, len(train_imgs), len(val_imgs), len(test_imgs)

ai_stats = split_class_images_multi(
    [RAW_PATH / "ai"],
    "ai"
)

real_stats = split_class_images_multi(
    [RAW_PATH / "real", RAW_PATH / "celeba_real"],
    "real"
)

print("===== SPLIT REPORT =====")
print(f"AI    → Total: {ai_stats[0]} | Train: {ai_stats[1]} | Val: {ai_stats[2]} | Test: {ai_stats[3]}")
print(f"REAL  → Total: {real_stats[0]} | Train: {real_stats[1]} | Val: {real_stats[2]} | Test: {real_stats[3]}")

===== SPLIT REPORT =====
AI    → Total: 4630 | Train: 3241 | Val: 694 | Test: 695
REAL  → Total: 55000 | Train: 38500 | Val: 8250 | Test: 8250


In [ ]:
# ==========================================
# MANIFEST DE INTERIM (paths + labels)
# ==========================================
# Objetivo:
# 1) Tener una tabla con rutas absolutas (o relativas) y etiquetas
# 2) Evitar reconstruir listados con glob en notebook 3
# 3) Asegurar reproducibilidad del set que se usará para tensores

import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
INTERIM_PATH = PROJECT_ROOT / "data" / "interim"

rows = []
label_map = {"real": 0, "ai": 1}

for split in ["train", "val", "test"]:
    for cls in ["real", "ai"]:
        folder = INTERIM_PATH / split / cls
        for p in folder.glob("*"):
            rows.append({
                "split": split,
                "class": cls,
                "label": label_map[cls],
                # guarda path como string
                "path": str(p)
            })

df_manifest = pd.DataFrame(rows)
manifest_path = INTERIM_PATH / "manifest_interim.csv"
df_manifest.to_csv(manifest_path, index=False)

print("Manifest guardado en:", manifest_path)
print("Filas:", len(df_manifest))
print(df_manifest.head())